In [1]:
from openai import OpenAI
import json
import requests

In [2]:
id = "7py38dci8xssgl"

In [3]:
runpod_url = f"https://{id}-8000.proxy.runpod.net"

In [28]:
runpod_url = "https://api-aipro.chatbaram.com/sllm"

In [4]:
health_url = f"{runpod_url}/health"

In [5]:
health_check_res = requests.get(health_url)
if health_check_res.status_code != 200:
    print("헬스 체크 실패")
    exit()

print(health_check_res.json())

{'status': 'ok'}


In [6]:
model_list_url = f"{runpod_url}/v1/models"

In [7]:
model_list_res = requests.get(model_list_url)

if model_list_res.status_code != 200:
    print("모델 목록 조회 실패")
    exit()

model_list = model_list_res.json()

print(model_list)

model_name = model_list['data'][0]['id']

{'object': 'list', 'data': [{'id': 'RedHatAI/gemma-4-31B-it-FP8-block', 'object': 'model', 'created': 1780636253, 'owned_by': 'vllm', 'root': 'RedHatAI/gemma-4-31B-it-FP8-block', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-8f4cd838da6c57cd', 'object': 'model_permission', 'created': 1780636253, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [8]:
chat_url = f"{runpod_url}/v1/chat/completions"

In [11]:
import base64

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [12]:
def build_messages(image_path: str, text: str) -> list:
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "gif": "image/gif"}
    mime_type = mime_map.get(ext, "image/jpeg")

    b64 = encode_image(image_path)

    print(f"{b64[:100]}...")

    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{b64}"}}
        ]
    }]



In [13]:
image_path = "D:\TEST\orchestrator_loop\image_data\\real_bill.jpg"

In [14]:
messages = build_messages(image_path, "영수증 내용을 마크다운으로 작성해줘")

/9j/2wBDAAMCAgICAgMCAgIDAwMDBAYEBAQEBAgGBgUGCQgKCgkICQkKDA8MCgsOCwkJDRENDg8QEBEQCgwSExIQEw8QEBD/2wBD...


In [15]:
payload = {
    "model" : "RedHatAI/gemma-4-31B-it-FP8-block",
    "messages": messages,
    "max_tokens": 4000,
    "stream": False
}

In [16]:
res = requests.post(chat_url, json=payload)

In [17]:
print(res.json())


{'id': 'chatcmpl-9cdfcfb041db80a7', 'object': 'chat.completion', 'created': 1780636319, 'model': 'RedHatAI/gemma-4-31B-it-FP8-block', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '이미지 속 영수증 내용을 마크다운 형식으로 정리해 드립니다.\n\n---\n\n# 🧾 영수증 내역\n\n**매장 정보**\n- **상호명:** 세븐일레븐 문정수정점 #18308\n- **전화번호:** 02-400-6307\n- **주소:** 서울특별시 송파구 동남로 8길 12 (문정동)\n- **웹사이트:** www.7eleven.co.kr\n\n**결제 정보**\n- **결제 일시:** 2020-06-09 (화) 20:59:47\n- **결제 수단:** 현금 (자진발급)\n\n**상세 내역**\n| 상품명 | 수량 | 금액 | 비고 |\n| :--- | :---: | :---: | :---: |\n| 라라스윗 바닐라 파인트 474ml | 1 | 6,900 | 행사 |\n| 라라스윗 초코 파인트 474ml | 1 | 6,900 | 행사 |\n| 비닐봉투 보증금 | 1 | 20 | |\n\n**금액 합계**\n- **공급가액:** 12,545원\n- **부가가치세:** 1,255원\n- **봉투보증금액:** 20원\n- **최종 합계 금액: ₩13,820**\n\n---', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': 106, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': No

In [18]:
payload['stream'] = True

res = requests.post(chat_url, json=payload)

is_reasoning = False

for line in res.iter_lines():
    if line:
        decoded = line.decode('utf-8')

        if decoded.startswith('data: '):
            data = decoded.split('data: ')[1]

            if data == '[DONE]':
                break;
            
            json_data = json.loads(data)
            
            tokens = json_data['choices'][0]['delta']

            if tokens.get('reasoning', None):
                if not is_reasoning: 
                    print("<REASONING>")
                    is_reasoning = True
                print(tokens['reasoning'], end='', flush=True)
            
            if tokens.get('content', None):
                if is_reasoning:
                    print("\n<RESPONSE>")
                    is_reasoning = False
                print(tokens['content'], end='', flush=True)

요청하신 영수증 내용을 마크다운 형식으로 정리해 드립니다.

***

# 🛒 세븐일레븐 영수증

**매장 정보**
* **점포명:** 문정수정점 #18308
* **전화번호:** 02-400-6307
* **주소:** 서울특별시 송파구 동남로 8길 12 (문정동)
* **홈페이지:** www.7eleven.co.kr

**결제 정보**
* **결제 일시:** 2020-06-09 (화) 20:59:47
* **결제 수단:** 현금 (자진발급)

**구매 내역**

| 상품명 | 수량 | 금액 | 비고 |
| :--- | :---: | :---: | :---: |
| 라라스윗 바닐라 파인트 474ml | 1 | 6,900원 | 행사 |
| 라라스윗 초코 파인트 474ml | 1 | 6,900원 | 행사 |
| 비닐봉투 보증금 20원 | 1 | 20원 | |

**금액 상세**
* **과세물품 가액:** 12,545원
* **부가세:** 1,255원
* **봉투보증금액:** 20원
* **합계 금액: ₩13,820**

**현금 결제액:** 20원 (봉투보증금 포함)

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api-aipro.chatbaram.com/sllm/v1",
    api_key="EMPTY"
)

response = client.chat.completions.create(
        model = "google/gemma-4-26B-A4B-it",
        messages = messages
    )

In [ ]:
print(response.choices[0].message.content)